In [ ]:
import adaptive_latents as al
import adaptive_latents.stimulation.photostimulation as ps
from adaptive_latents import NumpyTimedDataSource, Bubblewrap, AnimationManager, BWRun
from adaptive_latents.transforms.proSVD import proSVD
from adaptive_latents.transforms.utils import prosvd_data, clip
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

In [ ]:
# Specify the paths to the files
stim_file_path = '../workspace/datasets/fish/output_020424_ds1/stimmed.txt'
C_file_path = '../workspace/datasets/fish/output_020424_ds1/analysis_proc_C.txt'
photo_file_path = '../workspace/datasets/fish/output_020424_ds1/photostims.npy'

# Load the files
v_stim = np.loadtxt(stim_file_path) 
"""1st entry: frame number,
2nd entry: ignore,
3rd entry: angle of motion L,
4th entry: angle of motion R,
5th entry: timestamp,"""
C = np.loadtxt(C_file_path)#
"""Calcium imaging. 
1st entry is neuron ID,
2nd is time (frame)"""
photostim = np.load(photo_file_path)
"""1st entry: frame number,
2nd entry: counter of stims,
3rd entry: neuron ID,
4th entry: position X of neuron,
5th entry: position Y of neuron,"""

# extra note: Fs= 2.3 Hz

## Absolute distance between photostims

In [ ]:
C_visual= C[:, :1114]
C_photostim = C[:, 1114:]

In [ ]:
plt.imshow(C_photostim, aspect='auto')

In [ ]:
#Dimensionality reduction 
pro = proSVD(6, centering=True) # like the PCA() step
pro.run_on(C_visual) # like the fit_transform step
small_C=pro.project(C_photostim).T # like the transform step

In [ ]:
small_C.shape, C.shape, photostim.shape , C_photostim.shape

In [ ]:
def rank_neurons(stim, C, photostim):
    # Create DataFrame for stim
    stim_columns = ['frame_number', 'ignore', 'angle_of_motion_L', 'angle_of_motion_R', 'timestamp']
    stim_df = pd.DataFrame(stim, columns=stim_columns)

    # Create DataFrame for C, where each column after the first represents a neuron
    neuron_ids = [f'neuron_{i}' for i in range(C.shape[0])]
    time_points = [f'frame_{i}' for i in range(C.shape[1])]
    C_df = pd.DataFrame(C.T, columns=neuron_ids, index=time_points)

    # Create DataFrame for photostim
    photostim_columns = ['frame_number', 'stim_counter', 'neuron_ID', 'position_X', 'position_Y']
    photostim_df = pd.DataFrame(photostim, columns=photostim_columns)

    # Handle neurons with two trials in the data frame: 14 and 98 turn into 15 and 99 in the second trial
    neuron_position_counts = photostim_df.groupby('neuron_ID')['position_X'].count()

    # Filter neurons with more than 6 occurrences
    neurons_with_many_positions = neuron_position_counts[neuron_position_counts > 6].index

    # Filter the DataFrame to include only these neurons
    filtered_neurons_df = photostim_df[photostim_df['neuron_ID'].isin(neurons_with_many_positions)]

    # Function to modify the neuron ID for the second group of 5 occurrences
    def modify_neuron_ids(df):
        modified_ids = df.copy()
        neuron_groups = modified_ids.groupby('neuron_ID')
        
        for neuron_id, group in neuron_groups:
            # Split the group into sets of 5
            for i in range(1, len(group) // 5 + 1):
                start_idx = i * 5
                if start_idx < len(group):
                    modified_ids.loc[group.index[start_idx:start_idx + 5], 'neuron_ID'] = str(int(neuron_id) + 1)
        
        return modified_ids

    # Apply the function to modify the neuron IDs in the original DataFrame
    modified_photostim_df = modify_neuron_ids(photostim_df)
    # Drop the last row by index (we only have one stimulation for the last neuron)
    modified_photostim_df = modified_photostim_df.drop(modified_photostim_df.index[-1])

    # Ensure the frame_number and neuron_ID are of integer type
    modified_photostim_df['frame_number'] = modified_photostim_df['frame_number'].astype(int)
    modified_photostim_df['neuron_ID'] = modified_photostim_df['neuron_ID'].astype(int)

    # Sort the df by 'frame_number'
    modified_photostim_df = modified_photostim_df.sort_values(by='frame_number').reset_index(drop=True)

    return modified_photostim_df

In [ ]:
neurons_df=rank_neurons(v_stim, C, photostim) 
index_df=neurons_df.drop_duplicates(subset='neuron_ID', keep='first')
index_df

In [ ]:
#Calculating the distance matrix

# Extract and adjust indices based on 'frame_number' from index_df
indices = index_df['frame_number'].values - 1114

# Convert indices to integers
indices = indices.astype(int)

#Select the corresponding rows from C_small
selected_rows = small_C[indices]

#Calculate the pairwise Euclidean distances
distances = pdist(selected_rows, metric='euclidean')
distance_matrix = squareform(distances)

#Convert the distance matrix to a DataFrame with original indices
neuron_indices = index_df['neuron_ID'].values.astype(int)
distance_df = pd.DataFrame(distance_matrix, index=neuron_indices, columns=neuron_indices)

print(distance_df)


In [ ]:
# Extract distances from one index to the next
sequential_distances = []
for i in range(len(neuron_indices) - 1):
    index1 = neuron_indices[i]
    index2 = neuron_indices[i + 1]
    distance = distance_df.loc[index1, index2]
    sequential_distances.append((index1, index2, distance))

# Step 6: Convert to DataFrame
sequential_distances_df = pd.DataFrame(sequential_distances, columns=['Index1', 'Index2', 'Distance'])

#Sort by distance 
sequential_distances = sequential_distances_df.nlargest(24, 'Distance')

print(sequential_distances)

In [ ]:
##REMEMBER THIS IS IN 3D, AND THE RANKING IS IN 6D
ps.plot_photostim(156,'proSVD',C, photostim, neurons_df)


## Path distances between photostims

In [ ]:
# Subtract 1114 from frame_number using .loc
index_df.loc[:, 'frame_number'] = index_df['frame_number'] - 1114

# Reset the index of index_df to ensure integer indexing
index_df.reset_index(drop=True, inplace=True)

# Function to calculate the sum of Euclidean distances in 6D
def calculate_sum_of_distances(small_C, index_df):
    results = []

    # Iterate over the ranges defined by frame_number in index_df
    for i in range(len(index_df) - 1):
        start_frame = index_df.loc[i, 'frame_number']
        end_frame = index_df.loc[i + 1, 'frame_number']
        neuron_id = index_df.loc[i, 'neuron_ID']

        # Extract the rows of small_C between start_frame and end_frame
        subset = small_C[start_frame:end_frame, :]

        # Calculate the pairwise Euclidean distances
        distances = pdist(subset, metric='euclidean')

        # Sum the distances
        sum_distances = np.sum(distances)
        results.append({'neuron_ID': neuron_id, 'sum_distances': sum_distances})

    return results

# Calculate the sum of distances for each jump in frame_number
sum_of_distances = calculate_sum_of_distances(small_C, index_df)

# Convert results to a DataFrame for better visualization
results_df = pd.DataFrame(sum_of_distances)

# Print the results
print(results_df)


In [ ]:
# Rank the results based on the sum of distances
results_df['rank'] = results_df['sum_distances'].rank(ascending=False, method='min')

# Sort the DataFrame by rank
results_df.sort_values(by='rank', inplace=True)

results_df

## Distance off(from?)-manifold

We want to get the "distance from manifold" for the photostimulations. From ooutputs of proSVD we do:
Display equation: $$X\perp = X - Q\tilde{X}$$

In [ ]:
#Xtilde = smallC, Q = pro.Q, X = C_photostim
C_small_orthogonal = C_photostim - pro.Q @ small_C.T
#C_photostim.shape,  pro.Q.shape , small_C.T.shape

Note 1: To take percenatjes from _small_orth we would need to normalize all the way from C, because C_small_orth only tell you what amount of 'info' was 'not taken' in that input of the matrix
Note 2: Q.T takes you down dimensions when projecting, Q take syou up to the original space